# Regressão com Limitador — Projeto de Monografia

Este notebook contém a implementação computacional do projeto. A base de dados é mantida separadamente no Google Drive por seu volume e deve estar disponível ao usuário antes da execução.

### Execução
1. Abra o notebook no Google Colab.
2. Execute a primeira célula e autorize o acesso ao Google Drive.
3. Certifique-se de que a pasta `dataset_tratado` está em `Meu Drive`.
4. Execute as células em ordem.

Os arquivos produzidos são gravados em `dataset_tratado/resultados`.

> **Nota:** a base de dados não faz parte deste repositório. Consulte o `README.md` do GitHub para as instruções de acesso à base.

In [ ]:
import os
import re
import warnings
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from math import radians, sin, cos, sqrt, atan2

from sklearn.linear_model    import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    ConfusionMatrixDisplay,
)
from sklearn.pipeline        import Pipeline

from google.colab import drive

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.4f}".format)


# ─────────────────────────────────────────────────────────────────
# CÉLULA 2 — Montagem do Drive e Configurações Globais
# ─────────────────────────────────────────────────────────────────
# A base de dados permanece no Google Drive e não é versionada no GitHub.
# Estrutura esperada:
#
# Meu Drive/
# └── dataset_tratado/
#     ├── *_treated.csv
#     └── resultados/
#
# Se você utilizar outro nome/local, altere DATA_DIR abaixo.

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive", force_remount=True)
    DATA_DIR = Path("/content/drive/MyDrive/dataset_tratado")
else:
    # Execução fora do Colab: ajuste para o diretório local da base.
    DATA_DIR = Path("./dataset_tratado")

OUTPUT_DIR = DATA_DIR / "resultados"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Base de dados não encontrada em: {DATA_DIR}\n"
        "Verifique se a pasta dataset_tratado está disponível no Google Drive."
    )

csv_count = len(list(DATA_DIR.glob("*_treated.csv")))
if csv_count == 0:
    raise FileNotFoundError(
        f"Nenhum arquivo *_treated.csv foi encontrado em: {DATA_DIR}"
    )

print(f"Pasta de dados  : {DATA_DIR}")
print(f"Pasta de saída  : {OUTPUT_DIR}")
print(f"Arquivos CSV    : {csv_count}")

# ── Colunas numéricas que serão normalizadas pelo StandardScaler ──
SCALE_COLS = [
    "Speed",
    "GPS Satellites",
    "GSM Signal",
    "HDOP",
    "External Voltage",
    "Internal Battery",
    "Lat",
    "Lng",
    # Novas features — adicionadas dinamicamente
    "delta_tempo_seg",
    "delta_velocidade",
    "distancia_haversine_km",
    "velocidade_calculada_kmh",
]

# ── Limiares ──────────────────────────────────────────────────────
VOLT_THRESHOLD = 7.0

print(f"Pasta de dados  : {DATA_DIR}")
print(f"Pasta de saída  : {OUTPUT_DIR}")

# ─────────────────────────────────────────────────────────────────
# CÉLULA 3 — Funções de Feature Engineering
# ─────────────────────────────────────────────────────────────────

# ── 3.1  Distância de Haversine ───────────────────────────────────
def haversine_km(lat1: float, lon1: float,
                 lat2: float, lon2: float) -> float:
    """
    Calcula a distância geodésica entre dois pontos GPS (em km).
    Usa a fórmula de Haversine — adequada para distâncias curtas
    como as percorridas em intervalos de rastreamento veicular.
    """
    R = 6371.0  # Raio médio da Terra em km
    phi1, phi2 = radians(lat1), radians(lat2)
    dphi       = radians(lat2 - lat1)
    dlambda    = radians(lon2 - lon1)

    a = sin(dphi / 2) ** 2 + cos(phi1) * cos(phi2) * sin(dlambda / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c


def compute_haversine_series(lat: pd.Series,
                              lng: pd.Series) -> pd.Series:
    """
    Vetoriza o cálculo de Haversine para duas Series (linha i → i-1).
    Retorna uma Series com a distância em km entre pontos consecutivos.
    NaN na primeira linha (sem ponto anterior).
    """
    lat_r  = np.radians(lat.values.astype(float))
    lng_r  = np.radians(lng.values.astype(float))

    lat1   = lat_r[:-1];  lat2  = lat_r[1:]
    lng1   = lng_r[:-1];  lng2  = lng_r[1:]

    dphi   = lat2 - lat1
    dlam   = lng2 - lng1

    a = (np.sin(dphi / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin(dlam / 2) ** 2)

    # Clip para evitar NaN em sqrt de floats ligeiramente > 1
    a      = np.clip(a, 0, 1)
    c      = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    dist   = 6371.0 * c

    # Retorna como Series alinhada ao índice original (NaN no primeiro)
    result              = pd.Series(np.nan, index=lat.index, dtype=float)
    result.iloc[1:]     = dist
    return result


# ── 3.2  Flags de Eventos ────────────────────────────────────────
def build_event_flags(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cria indicadores binários a partir das colunas RC_* presentes
    no DataFrame.

    tem_tamper    — qualquer coluna com 'Tamper' ou 'Violation' = 1
    tem_jammer    — qualquer coluna com 'Jammer' ou 'Jamming'  = 1
    queda_bateria — External Voltage < limiar
    """
    cols = df.columns.tolist()

    # Tamper / Violation
    tamper_cols = [c for c in cols
                   if re.search(r"tamper|violation", c, re.I)]
    df["tem_tamper"] = (
        df[tamper_cols].any(axis=1).astype(int)
        if tamper_cols else 0
    )

    # Jammer / Jamming
    jammer_cols = [c for c in cols
                   if re.search(r"jammer|jamming", c, re.I)]
    df["tem_jammer"] = (
        df[jammer_cols].any(axis=1).astype(int)
        if jammer_cols else 0
    )

    # Queda de bateria / tensão externa
    if "External Voltage" in df.columns:
        ext = pd.to_numeric(df["External Voltage"], errors="coerce")
        df["queda_bateria"] = (ext < VOLT_THRESHOLD).astype(int)
    else:
        df["queda_bateria"] = 0

    return df


# ── 3.3  Pipeline de Feature Engineering por Arquivo ─────────────
def engineer_features(df: pd.DataFrame,
                       tracker_name: str = "") -> pd.DataFrame:
    """
    Aplica toda a engenharia de atributos temporais em um único
    DataFrame de rastreador, mantendo a ordem cronológica.

    Parâmetros
    ----------
    df           : DataFrame já tratado (saída do pipeline anterior)
    tracker_name : Nome do arquivo (usado como ID na coluna 'rastreador')

    Retorna
    -------
    DataFrame enriquecido com as novas features.
    """
    df = df.copy()

    # ── Garantir ordenação temporal ───────────────────────────────
    if "Hora" in df.columns:
        df.sort_values("Hora", kind="mergesort", inplace=True)
    df.reset_index(drop=True, inplace=True)

    # ── 1. Delta de Tempo (segundos entre registros) ──────────────
    #    Como temos apenas a hora (int 0-23), calculamos a diferença
    #    em segundos multiplicando por 3600. Para precisão real, use
    #    o datetime completo; aqui é a melhor aproximação disponível.
    if "Hora" in df.columns:
        hora_num = pd.to_numeric(df["Hora"], errors="coerce")
        delta_h  = hora_num.diff()          # diferença em horas (pode ser negativa p/ virada de dia)
        # Tratar virada de dia (ex: 23 → 0): adicionar 24 se negativo
        delta_h  = delta_h.apply(lambda x: x + 24 if (pd.notna(x) and x < 0) else x)
        df["delta_tempo_seg"] = (delta_h * 3600).round(2)
    else:
        df["delta_tempo_seg"] = np.nan

    # ── 2. Variação de Velocidade ─────────────────────────────────
    if "Speed" in df.columns:
        speed_num = pd.to_numeric(df["Speed"], errors="coerce")
        df["delta_velocidade"] = speed_num.diff()
    else:
        df["delta_velocidade"] = np.nan

    # ── 3. Distância Haversine (km) ───────────────────────────────
    if {"Lat", "Lng"}.issubset(df.columns):
        lat_num = pd.to_numeric(df["Lat"], errors="coerce")
        lng_num = pd.to_numeric(df["Lng"], errors="coerce")

        # Só calcula onde ambas as linhas consecutivas são válidas
        valid_mask = lat_num.notna() & lng_num.notna()
        dist_series = pd.Series(np.nan, index=df.index, dtype=float)

        if valid_mask.sum() > 1:
            # Trabalhar com sub-série válida para evitar propagação de NaN
            lat_clean  = lat_num.where(valid_mask)
            lng_clean  = lng_num.where(valid_mask)
            dist_series = compute_haversine_series(lat_clean, lng_clean)

        df["distancia_haversine_km"] = dist_series.values
    else:
        df["distancia_haversine_km"] = np.nan

    # ── 4. Velocidade Calculada (km/h) ────────────────────────────
    #    Distância (km) / Tempo (seg) * 3600 = km/h
    #    Guard: tempo deve ser > 0 para evitar divisão por zero
    tempo_seg = pd.to_numeric(
        df.get("delta_tempo_seg", pd.Series(np.nan, index=df.index)),
        errors="coerce",
    )
    dist_km = df.get(
        "distancia_haversine_km",
        pd.Series(np.nan, index=df.index),
    )

    df["velocidade_calculada_kmh"] = np.where(
        (tempo_seg > 0) & tempo_seg.notna() & dist_km.notna(),
        (dist_km / tempo_seg) * 3600,
        np.nan,
    )

    # ── 5. Flags de Eventos ───────────────────────────────────────
    df = build_event_flags(df)

    # ── 6. Identificador do rastreador ────────────────────────────
    df["rastreador"] = tracker_name

    # ── 7. Tratar NaN / Inf residuais ────────────────────────────
    #    Substituir Inf por NaN primeiro, depois preencher NaN
    #    nas colunas de delta com 0 (sem dado anterior = sem variação)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    fill_zero_cols = [
        "delta_tempo_seg",
        "delta_velocidade",
        "distancia_haversine_km",
        "velocidade_calculada_kmh",
    ]
    for col in fill_zero_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    return df



# ─────────────────────────────────────────────────────────────────
# CÉLULA 4 — Carregamento e Aplicação da Engenharia de Atributos
# ─────────────────────────────────────────────────────────────────

def load_and_engineer_all(data_dir: Path, max_files: int = None) -> pd.DataFrame:
    """
    Lê todos os CSVs tratados, aplica a engenharia de atributos
    em cada um individualmente e concatena em um DataFrame global.
    """
    csv_files = sorted(data_dir.glob("*_treated.csv"))

    if not csv_files:
        # Fallback: qualquer CSV na pasta
        csv_files = sorted(data_dir.glob("*.csv"))

    if not csv_files:
        raise FileNotFoundError(f"Nenhum CSV encontrado em {data_dir}")

    if max_files is not None and max_files > 0:
        csv_files = csv_files[:max_files]

    frames  = []
    erros   = []
    total   = len(csv_files)

    print(f"CARREGANDO {total} ARQUIVO(S) + FEATURE ENGINEERING")

    for idx, fp in enumerate(csv_files, 1):
        try:
            df_raw = pd.read_csv(fp, sep=";", encoding="utf-8", low_memory=False)

            # Remover linhas completamente nulas
            df_raw.dropna(how="all", inplace=True)

            if df_raw.empty:
                print(f"[{idx:>3}/{total}] {fp.name} — vazio, ignorado.")
                continue

            df_eng = engineer_features(df_raw, tracker_name=fp.stem)
            frames.append(df_eng)

            n_fraude = int(df_eng.get("Fraude", pd.Series(0)).sum())
            print(
                f"[{idx:>3}/{total}] {fp.name:<45} "
                f"{len(df_eng):>6} linhas | Fraudes: {n_fraude}"
            )

        except Exception as exc:
            erros.append((fp.name, str(exc)))
            print(f"[{idx:>3}/{total}] {fp.name} — ERRO: {exc}")

    if not frames:
        raise ValueError("Nenhum arquivo pôde ser carregado.")

    df_global = pd.concat(frames, axis=0, ignore_index=True)

    print(f"Concluído: {len(frames)} arquivos carregados | "
          f"{len(erros)} erros")
    print(f"DataFrame global: {df_global.shape[0]:,} linhas × "
          f"{df_global.shape[1]} colunas")
    if erros:
        print(f"Arquivos com erro:")
        for nome, msg in erros:
            print(f" • {nome}: {msg}")

    return df_global


# ── EXECUTAR ──────────────────────────────────────────────────────
df_global = load_and_engineer_all(DATA_DIR, max_files=80)

# Verificação rápida
print("Primeiras linhas do DataFrame global:")
display(df_global.head(3))

print(f"\nDistribuição da Label 'Fraude':")
if "Fraude" in df_global.columns:
    vc = df_global["Fraude"].value_counts()
    print(vc)
    print(f"Taxa de fraude: {vc.get(1,0) / len(df_global) * 100:.2f}%")

# ─────────────────────────────────────────────────────────────────
# CÉLULA 5 — Preparação para Modelagem
# ─────────────────────────────────────────────────────────────────

def prepare_model_data(df: pd.DataFrame):
    """
    Separa features (X) e target (y), define quais colunas
    são normalizadas, e realiza o triple split estratificado:
    60% treino | 20% validação | 20% teste.

    Retorna
    -------
    X_train, X_val, X_test,
    y_train, y_val, y_test,
    feature_names, scaler
    """
    # ── Target ────────────────────────────────────────────────────
    if "Fraude" not in df.columns:
        raise ValueError("Coluna 'Fraude' não encontrada no DataFrame.")

    y = df["Fraude"].astype(int)

    # ── Remover colunas não-feature ───────────────────────────────
    drop_cols = ["Fraude", "rastreador", "Hora"]
    X = df.drop(columns=[c for c in drop_cols if c in df.columns])

    # ── Garantir que X é todo numérico ───────────────────────────
    X = X.apply(pd.to_numeric, errors="coerce")
    X.replace([np.inf, -np.inf], np.nan, inplace=True)

    # Preencher NaN com mediana de cada coluna (robusto a outliers)
    X.fillna(X.median(numeric_only=True), inplace=True)

    feature_names = X.columns.tolist()
    print(f"  → {len(feature_names)} features para o modelo.")

    # ── Triple Split: 60 / 20 / 20 ────────────────────────────────
    # Passo 1: treino (60%) + temp (40%)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y,
        test_size=0.40,
        random_state=42,
        stratify=y,
    )
    # Passo 2: validação (50% do temp = 20% total) + teste (20%)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.50,
        random_state=42,
        stratify=y_temp,
    )

    print(f"\n  Tamanhos dos conjuntos (total: {len(y):,} registros):")
    for nome, yy in [("Treino", y_train), ("Validação", y_val), ("Teste", y_test)]:
        n1 = yy.sum()
        pct = n1 / len(yy) * 100
        print(f"    {nome:<12}: {len(yy):>7,} registros | "
              f"Fraudes: {n1:>5,} ({pct:.2f}%) Kishan")

    # ── Normalização ──────────────────────────────────────────────
    # Aplica StandardScaler apenas nas colunas numéricas contínuas
    scale_cols_present = [c for c in SCALE_COLS if c in feature_names]
    other_cols         = [c for c in feature_names if c not in scale_cols_present]

    scaler = StandardScaler()

    def scale_df(X_df, fit=False):
        X_out = X_df.copy()
        if scale_cols_present:
            if fit:
                X_out[scale_cols_present] = scaler.fit_transform(
                    X_df[scale_cols_present]
                )
            else:
                X_out[scale_cols_present] = scaler.transform(
                    X_df[scale_cols_present]
                )
        return X_out

    X_train = scale_df(X_train, fit=True)
    X_val   = scale_df(X_val,   fit=False)
    X_test  = scale_df(X_test,  fit=False)

    print(f"\n  StandardScaler aplicado em {len(scale_cols_present)} colunas.")

    return (
        X_train, X_val, X_test,
        y_train, y_val, y_test,
        feature_names, scaler
    )


print("PREPARANDO DADOS PARA MODELAGEM...")

(X_train, X_val, X_test,
 y_train, y_val, y_test,
 feature_names, scaler) = prepare_model_data(df_global)

# ─────────────────────────────────────────────────────────────────
# CÉLULA 6 — Treinamento da Regressão Logística
# ─────────────────────────────────────────────────────────────────

def train_logistic_model(X_train, y_train,
                          X_val,   y_val):
    """
    Treina LogisticRegression com class_weight='balanced' para
    lider com o desbalanceamento de classes.

    Avalia no conjunto de validação e imprime o relatório.
    """
    model = LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        solver="lbfgs",         # eficiente para conjuntos médios
        C=1.0,                  # regularização padrão (ajustável)
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)
    print("Modelo treinado com sucesso!\n")

    # ── Avaliação no Conjunto de Treino ────────────────────────
    y_train_pred  = model.predict(X_train)
    y_train_proba = model.predict_proba(X_train)[:, 1]
    roc_train     = roc_auc_score(y_train, y_train_proba)

    print("AVALIAÇÃO — CONJUNTO DE TREINO")
    print(classification_report(y_train, y_train_pred,
                                  target_names=["Normal (0)", "Fraude (1)"]))
    print(f"ROC-AUC (Treino): {roc_train:.4f}")

    # ── Avaliação no Conjunto de Validação ────────────────────────
    y_val_pred  = model.predict(X_val)
    y_val_proba = model.predict_proba(X_val)[:, 1]
    roc_val     = roc_auc_score(y_val, y_val_proba)

    print("AVALIAÇÃO — CONJUNTO DE VALIDAÇÃO")
    print(classification_report(y_val, y_val_pred,
                                  target_names=["Normal (0)", "Fraude (1)"]))
    print(f"ROC-AUC (Validação): {roc_val:.4f}")

    return model, y_train_pred, y_val_pred


model, y_train_pred, y_val_pred = train_logistic_model(X_train, y_train, X_val, y_val)

# ─────────────────────────────────────────────────────────────────
# CÉLULA 7 — Avaliação Final no Conjunto de Teste
# ─────────────────────────────────────────────────────────────────

y_test_pred  = model.predict(X_test)
y_test_proba = model.predict_proba(X_test)[:, 1]
roc_test     = roc_auc_score(y_test, y_test_proba)

print("AVALIAÇÃO FINAL — CONJUNTO DE TESTE")
print(classification_report(y_test, y_test_pred,
                              target_names=["Normal (0)", "Fraude (1)"]))
print(f"ROC-AUC (Teste): {roc_test:.4f}")

# ─────────────────────────────────────────────────────────────────
# CÉLULA 8 — Visualizações Globais
# ─────────────────────────────────────────────────────────────────

def plot_global_metrics(y_true, y_pred, y_proba,
                         roc_auc_score_val: float,
                         set_name: str = "Teste"):
    """
    Gera dois gráficos lado a lado:
      - Matriz de Confusão (normalizada)
      - Curva ROC-AUC
    """
    fig = plt.figure(figsize=(16, 6), facecolor="#FFFFFF") # Changed to white
    gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    # Paleta para tema claro
    FG   = "#333333"   # Dark gray for text
    ACC1 = "#008060"   # Darker green for contrast on white
    ACC2 = "#CC3300"   # Darker red for contrast on white
    BG   = "#FFFFFF"   # White background
    GRID = "#CCCCCC"   # Light gray for grid/spines

    # ── Matriz de Confusão ────────────────────────────────────────
    cm      = confusion_matrix(y_true, y_pred, normalize="true")
    cm_disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                      display_labels=["Normal", "Fraude"])
    cm_disp.plot(
        ax=ax1,
        colorbar=False,
        cmap=plt.cm.Greens, # Use a suitable cmap for white background, e.g., Greens/Reds
    )
    ax1.set_title(f"Matriz de Confusão\n(Normalizada — {set_name})",
                  color=FG, fontsize=13, pad=12, fontweight="bold")
    ax1.set_facecolor(BG)
    ax1.tick_params(colors=FG)
    ax1.xaxis.label.set_color(FG)
    ax1.yaxis.label.set_color(FG)
    for spine in ax1.spines.values():
        spine.set_edgecolor(GRID)

    # ConfusionMatrixDisplay automatically handles text color for contrast.
    # We only need to adjust font size and weight.
    for text in cm_disp.text_.ravel():
        text.set_fontsize(14)
        text.set_fontweight("bold")

    # ── Curva ROC ─────────────────────────────────────────────────
    fpr, tpr, _ = roc_curve(y_true, y_proba)

    ax2.set_facecolor(BG)
    ax2.plot(fpr, tpr, color=ACC1, lw=2.5,
             label=f"Modelo (AUC = {roc_auc_score_val:.3f})")
    ax2.plot([0, 1], [0, 1], color=GRID,
             lw=1.5, linestyle="--", label="Aleatório (AUC = 0.5)")
    ax2.fill_between(fpr, tpr, alpha=0.12, color=ACC1)

    # Ponto ótimo (maximiza TPR - FPR)
    j_scores = tpr - fpr
    opt_idx  = np.argmax(j_scores)
    ax2.scatter(fpr[opt_idx], tpr[opt_idx],
                color=ACC2, s=100, zorder=5,
                label=f"Ponto Ótimo (J={j_scores[opt_idx]:.3f})")

    ax2.set_xlabel("Taxa de Falso Positivo (FPR)", color=FG)
    ax2.set_ylabel("Taxa de Verdadeiro Positivo (TPR)", color=FG)
    ax2.set_title(f"Curva ROC-AUC\n({set_name})",
                  color=FG, fontsize=13, pad=12, fontweight="bold")
    ax2.tick_params(colors=FG)
    ax2.legend(facecolor=BG, edgecolor=GRID,
               labelcolor=FG, fontsize=9)
    for spine in ax2.spines.values():
        spine.set_edgecolor(GRID)

    fig.suptitle(
        "DETECÇÃO DE FRAUDE EM RASTREADORES — REGRESSÃO LOGÍSTICA",
        color=FG, fontsize=15, fontweight="bold", y=1.02,
    )

    plt.savefig(OUTPUT_DIR / "global_metrics.png",
                dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    print(f"Gráfico salvo em: {OUTPUT_DIR / 'global_metrics.png'}")


plot_global_metrics(y_test, y_test_pred, y_test_proba, roc_test)

# ── Gráfico da função sigmoide ───────────────────────────────────
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

x = np.linspace(-10, 10, 400)
y = sigmoid(x)

fig, ax = plt.subplots(figsize=(8, 5), facecolor="#FFFFFF") # Changed to white
ax.set_facecolor("#FFFFFF") # Changed to white

ax.plot(x, y, color="#008060", lw=2) # Darker green for visibility
ax.axvline(0, color="#999999", linestyle="--", lw=1) # Darker gray
ax.axhline(0.5, color="#999999", linestyle="--", lw=1) # Darker gray

ax.set_title("Função Sigmoide (Ativação Logística)", color="#333333", fontsize=13, pad=12, fontweight="bold") # Dark gray text
ax.set_xlabel("Input (logit)", color="#333333") # Dark gray text
ax.set_ylabel("Output (Probabilidade)", color="#333333") # Dark gray text
ax.tick_params(colors="#333333") # Dark gray ticks

for spine in ax.spines.values():
    spine.set_edgecolor("#CCCCCC") # Light gray spines

plt.grid(True, linestyle=":", alpha=0.6, color="#CCCCCC") # Light gray grid
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sigmoid_function.png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Gráfico salvo em: {OUTPUT_DIR / 'sigmoid_function.png'}")

# ─────────────────────────────────────────────────────────────────
# CÉLULA 9 — Importância das Features (Coeficientes)
# ─────────────────────────────────────────────────────────────────

def plot_feature_importance(model, feature_names: list, top_n: int = 25):
    """
    Plota os top N coeficientes da Regressão Logística.
    Coeficientes positivos → aumentam risco de fraude.
    Coeficientes negativos → reduzem risco de fraude.
    """
    coef_df = (
        pd.DataFrame({
            "feature"    : feature_names,
            "coeficiente": model.coef_[0],
        })
        .assign(abs_coef=lambda d: d["coeficiente"].abs())
        .sort_values("abs_coef", ascending=False)
        .head(top_n)
        .sort_values("coeficiente")
    )

    # Adjusted colors for white background
    colors = ["#CC3300" if c > 0 else "#008060" # Darker red/green
              for c in coef_df["coeficiente"]]

    fig, ax = plt.subplots(figsize=(10, max(6, top_n * 0.32)),
                            facecolor="#FFFFFF") # Changed to white
    ax.set_facecolor("#FFFFFF") # Changed to white

    bars = ax.barh(
        coef_df["feature"], coef_df["coeficiente"],
        color=colors, edgecolor="#FFFFFF", linewidth=0.5) # White edge
    ax.axvline(0, color="#999999", linewidth=1.2, linestyle="--") # Darker gray

    ax.set_title(f"Top {top_n} Features — Coeficientes da Regressão Logística",
                 color="#333333", fontsize=12, fontweight="bold", pad=12) # Dark gray text
    ax.set_xlabel("Coeficiente (Impacto no Log-Odds de Fraude)",
                  color="#333333") # Dark gray text
    ax.tick_params(colors="#333333", labelsize=9) # Dark gray ticks

    from matplotlib.patches import Patch
    legend = [
        Patch(facecolor="#CC3300", label="↑ Aumenta risco de fraude"), # Darker red
        Patch(facecolor="#008060", label="↓ Reduz risco de fraude"), # Darker green
    ]
    ax.legend(handles=legend, facecolor="#FFFFFF", # White background
              edgecolor="#CCCCCC", labelcolor="#333333", fontsize=9) # Light gray edge, dark gray text

    for spine in ax.spines.values():
        spine.set_edgecolor("#CCCCCC") # Light gray spines

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "feature_importance.png",
                dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.show()
    print(f"Gráfico salvo em: {OUTPUT_DIR / 'feature_importance.png'}")


plot_feature_importance(model, feature_names)

# ─────────────────────────────────────────────────────────────────
# CÉLULA 10 — Relatório Individual por Rastreador
# ─────────────────────────────────────────────────────────────────

def individual_tracker_report(data_dir: Path,
                               model: LogisticRegression,
                               scaler: StandardScaler,
                               feature_names: list,
                               scale_cols: list) -> pd.DataFrame:
    """
    Percorre os arquivos originais, aplica a engenharia de atributos,
    gera predições e imprime o relatório individual por rastreador.

    Retorna um DataFrame consolidado com os resultados.
    """
    csv_files   = sorted(data_dir.glob("*_treated.csv"))
    if not csv_files:
        csv_files = sorted(data_dir.glob("*.csv"))

    scale_cols_present = [c for c in scale_cols if c in feature_names]

    relatorio_rows = []

    print(f"RELATÓRIO INDIVIDUAL POR RASTREADOR ({len(csv_files)} arquivos)")
    print(
        f"{'ARQUIVO':<40} | {'TOTAL':>7} | "
        f"{'FRAUDES':>8} | {'RISCO MÉDIO':>12}"
    )

    for fp in csv_files:
        try:
            df_raw = pd.read_csv(fp, sep=";", encoding="utf-8", low_memory=False)
            df_raw.dropna(how="all", inplace=True)

            if df_raw.empty:
                continue

            # Engenharia de atributos
            df_eng = engineer_features(df_raw, tracker_name=fp.stem)

            # Preparar X com as mesmas features do treino
            drop_cols = ["Fraude", "rastreador", "Hora"]
            X_inf = df_eng.drop(
                columns=[c for c in drop_cols if c in df_eng.columns]
            )
            X_inf = X_inf.apply(pd.to_numeric, errors="coerce")
            X_inf.replace([np.inf, -np.inf], np.nan, inplace=True)

            # Alinhar colunas — adicionar 0 para features ausentes
            for feat in feature_names:
                if feat not in X_inf.columns:
                    X_inf[feat] = 0
            X_inf = X_inf[feature_names]

            # Preencher NaN com 0 (conservador para inferência)
            X_inf.fillna(0, inplace=True)

            # Normalizar as mesmas colunas do treino
            X_inf_scaled = X_inf.copy()
            if scale_cols_present:
                X_inf_scaled[scale_cols_present] = scaler.transform(
                    X_inf[scale_cols_present]
                )

            # Predição
            probas        = model.predict_proba(X_inf_scaled)[:, 1]
            preds         = model.predict(X_inf_scaled)
            risco_medio   = float(np.mean(probas) * 100)
            n_fraudes_pred= int(preds.sum())
            total         = len(df_eng)

            # Flag de alerta
            alerta = ""
            if risco_medio >= 50:
                alerta = "ALTO RISCO"
            elif risco_medio >= 20:
                alerta = "RISCO MÉDIO"

            print(
                f"{fp.stem:<40} | {total:>7,} | "
                f"{n_fraudes_pred:>8,} | {risco_medio:>10.2f}%  {alerta}"
            )

            relatorio_rows.append({
                "arquivo"          : fp.name,
                "rastreador"       : fp.stem,
                "total_registros"  : total,
                "fraudes_preditas" : n_fraudes_pred,
                "risco_medio_pct"  : round(risco_medio, 4),
                "alerta"           : alerta.strip(),
            })

        except Exception as exc:
            print(f"ERRO em '{fp.name}': {exc}")


    df_report = pd.DataFrame(relatorio_rows).sort_values(
        "risco_medio_pct", ascending=False
    )

    # Salvar relatório
    report_path = OUTPUT_DIR / "relatorio_rastreadores.csv"
    df_report.to_csv(report_path, index=False, sep=";", encoding="utf-8")
    print(f"Relatório salvo em: {report_path}")

    return df_report


# ── EXECUTAR ──────────────────────────────────────────────────────
df_report = individual_tracker_report(
    data_dir      = DATA_DIR,
    model         = model,
    scaler        = scaler,
    feature_names = feature_names,
    scale_cols    = SCALE_COLS,
)

# ─────────────────────────────────────────────────────────────────
# CÉLULA 11 — Visualização: Ranking de Risco por Rastreador
# ─────────────────────────────────────────────────────────────────

def plot_tracker_risk_ranking(df_report: pd.DataFrame,
                               top_n: int = 20):
    """
    Plota um gráfico horizontal com o top N rastreadores
    com maior probabilidade média de risco.
    """
    if df_report.empty:
        print("Relatório vazio, nenhum gráfico gerado.")
        return

    top = df_report.head(top_n).copy()
    top["rastreador_label"] = top["rastreador"].str.replace("_treated", "", regex=False)

    # Escala de cores por nível de risco - Adjusted for white background
    def risk_color(pct):
        if pct >= 50:  return "#CC3300"   # Darker red — alto
        if pct >= 20:  return "#FF7F00"   # Darker orange  — médio
        return                "#008060"   # Darker green    — baixo

    bar_colors = [risk_color(p) for p in top["risco_medio_pct"]]

    fig, ax = plt.subplots(
        figsize=(12, max(6, top_n * 0.42)),
        facecolor="#FFFFFF", # Changed to white
    )
    ax.set_facecolor("#FFFFFF") # Changed to white

    bars = ax.barh(
        top["rastreador_label"][::-1],
        top["risco_medio_pct"][::-1],
        color=list(reversed(bar_colors)),
        edgecolor="#FFFFFF", linewidth=0.5, # White edge
    )

    # Rótulos internos
    for bar, val in zip(bars, reversed(top["risco_medio_pct"].tolist())):
        ax.text(
            bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%",
            va="center", color="#333333", fontsize=8.5, # Dark gray text
        )

    ax.set_xlabel("Probabilidade Média de Risco (%)", color="#333333") # Dark gray text
    ax.set_title(
        f"Top {top_n} Rastreadores — Risco Médio de Fraude",
        color="#333333", fontsize=13, fontweight="bold", pad=12, # Dark gray text
    )
    ax.axvline(50, color="#CC3300", lw=1.2, ls="--", alpha=0.6, # Darker red
               label="Limiar: 50% (Alto Risco)")
    ax.axvline(20, color="#FF7F00", lw=1.2, ls="--", alpha=0.6, # Darker orange
               label="Limiar: 20% (Risco Médio)")

    ax.tick_params(colors="#333333", labelsize=9) # Dark gray ticks
    ax.legend(facecolor="#FFFFFF", edgecolor="#CCCCCC", # White background, light gray edge
              labelcolor="#333333", fontsize=9) # Dark gray text
    for spine in ax.spines.values():
        spine.set_edgecolor("#CCCCCC") # Light gray spines

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "ranking_risco_rastreadores.png",
                dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.show()
    print(f"Gráfico salvo em: {OUTPUT_DIR / 'ranking_risco_rastreadores.png'}")


plot_tracker_risk_ranking(df_report, top_n=20)

# ─────────────────────────────────────────────────────────────────
# CÉLULA 12 — Salvar Modelo e Scaler (joblib)
# ─────────────────────────────────────────────────────────────────
import joblib

model_path  = OUTPUT_DIR / "logistic_model.pkl"
scaler_path = OUTPUT_DIR / "scaler.pkl"
feats_path  = OUTPUT_DIR / "feature_names.txt"

joblib.dump(model,  model_path)
joblib.dump(scaler, scaler_path)

with open(feats_path, "w") as f:
    f.write("\n".join(feature_names))

print(f"Modelo salvo em  : {model_path}")
print(f"Scaler salvo em  : {scaler_path}")
print(f"Features salvas em: {feats_path}")
print("\n  Para carregar em outro notebook:")
print("  model  = joblib.load('logistic_model.pkl')")
print("  scaler = joblib.load('scaler.pkl')")

In [ ]:
# A função `get_execution_results` não é padrão e causou um erro.
# A célula anterior (c5e4362a) já salvou os relatórios de classificação como imagens PNG
# e imprimiu as confirmações. Esta célula não é necessária.
print('A execução da célula c5e4362a já gerou e salvou os relatórios de classificação em PNG.')
print('Verifique os arquivos no diretório de saída para os gráficos gerados.')

### Visualizações Adicionais: Tabelas de Avaliação e Relatório por Rastreador em PNG

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import classification_report

def plot_text_report(title: str, report_str: str, filename: str):
    """
    Cria uma imagem PNG a partir de uma string de relatório.
    """
    fig, ax = plt.subplots(figsize=(10, 6), facecolor="#FFFFFF")
    ax.set_facecolor("#FFFFFF")
    ax.text(0.01, 0.99,
            report_str,
            family='monospace',
            fontsize=12,
            verticalalignment='top',
            bbox=dict(boxstyle='square,pad=0.5', fc="#F0F0F0", ec="none"))
    ax.set_title(title, color="#333333", fontsize=14, fontweight="bold", pad=20)
    ax.axis('off') # Remove os eixos

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename,
                dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    print(f"Gráfico salvo em: {OUTPUT_DIR / filename}")


# Gerar e plotar o relatório de classificação do conjunto de Treino
report_train = classification_report(y_train, y_train_pred,
                                     target_names=["Normal (0)", "Fraude (1)"])
plot_text_report("Relatório de Classificação — Treino", report_train, "classification_report_train.png")

# Gerar e plotar o relatório de classificação do conjunto de Validação
report_val = classification_report(y_val, y_val_pred,
                                   target_names=["Normal (0)", "Fraude (1)"])
plot_text_report("Relatório de Classificação — Validação", report_val, "classification_report_val.png")

# Gerar e plotar o relatório de classificação do conjunto de Teste
report_test = classification_report(y_test, y_test_pred,
                                    target_names=["Normal (0)", "Fraude (1)"])
plot_text_report("Relatório de Classificação — Teste", report_test, "classification_report_test.png")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_dataframe_as_image(df: pd.DataFrame, title: str, filename: str):
    """
    Cria uma imagem PNG a partir de um DataFrame.
    """
    # Aumenta a largura da figura para dar mais espaço para as colunas
    fig, ax = plt.subplots(figsize=(18, min(20, len(df) * 0.4 + 2)), facecolor="#FFFFFF")
    ax.set_facecolor("#FFFFFF")
    ax.axis('off')

    # Renderizar o DataFrame como uma tabela
    table = ax.table(cellText=df.values,
                     colLabels=df.columns,
                     loc='center',
                     cellLoc='center')

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    # Tenta ajustar automaticamente a largura das colunas
    table.auto_set_column_width(col=list(range(len(df.columns))))
    table.scale(1.2, 1.2) # Mantém o escalonamento para espaçamento

    ax.set_title(title, color="#333333", fontsize=14, fontweight="bold", pad=20)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename,
                dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    print(f"Gráfico salvo em: {OUTPUT_DIR / filename}")


# Plotar o relatório individual por rastreador como imagem
plot_dataframe_as_image(df_report.head(20).round(2), # Limitar para melhor visualização na imagem
                        "Relatório Individual por Rastreador (Top 20)",
                        "individual_tracker_report.png")

## Arquivos gerados

Após a execução, os principais resultados são gravados na pasta `resultados/` do diretório da base, incluindo gráficos, relatório por rastreador, modelo de regressão, scaler e lista de atributos.

Os resultados gerados durante a execução não são enviados automaticamente ao GitHub. Eles permanecem no Google Drive utilizado na execução.